In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

In [3]:
np.random.seed(42) # Para que los resultados sean reproducibles

In [4]:
productos = {
    1: {"nombre": "Coca-Cola 600ml", "precio": 18.00, "base_demanda": 15},
    2: {"nombre": "Sabritas Original", "precio": 17.00, "base_demanda": 10},
    3: {"nombre": "Pan Bimbo", "precio": 45.00, "base_demanda": 5},
    4: {"nombre": "Leche Lala 1L", "precio": 28.00, "base_demanda": 8},
}

In [5]:
fecha_inicio = datetime(2024, 1, 1)
fecha_fin = datetime(2024, 6, 30)
fechas = pd.date_range(fecha_inicio, fecha_fin, freq='D')

print(f"Total de dias a generar: {len(fechas)}")
print(fechas[:5])

Total de dias a generar: 182
DatetimeIndex(['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04',
               '2024-01-05'],
              dtype='datetime64[us]', freq='D')


In [6]:
registros_ventas = []

for fecha in fechas:
    for producto_id, info in productos.items():
        es_fin_semana = fecha.weekday() >= 5  # 5=sábado, 6=domingo
        
        # Demanda base + boost de fin de semana + ruido aleatorio
        demanda = info["base_demanda"]
        if es_fin_semana:
            demanda = demanda * 1.4  # 40% más ventas en fin de semana
        
        # Ruido aleatorio con distribución normal
        cantidad = int(np.random.normal(loc=demanda, scale=demanda*0.3))
        cantidad = max(0, cantidad)  # no permitir cantidades negativas
        
        if cantidad > 0:
            registros_ventas.append({
                "producto_id": producto_id,
                "fecha": fecha,
                "cantidad": cantidad,
                "precio_unitario": info["precio"]
            })

df_ventas_sim = pd.DataFrame(registros_ventas)
print(df_ventas_sim.shape)
df_ventas_sim.head(10)

(726, 4)


,producto_id,fecha,cantidad,precio_unitario
0,1,2024-01-01,17,18.0
1,2,2024-01-01,9,17.0
2,3,2024-01-01,5,45.0
3,4,2024-01-01,11,28.0
4,1,2024-01-02,13,18.0
5,2,2024-01-02,9,17.0
6,3,2024-01-02,7,45.0
7,4,2024-01-02,9,28.0
8,1,2024-01-03,12,18.0
9,2,2024-01-03,11,17.0


In [ ]:
df_ventas_sim['fecha'] = pd.to_datetime(df_ventas_sim['fecha'])
df_ventas_sim['es_fin_semana'] = df_ventas_sim['fecha'].dt.weekday >= 5

print(df_ventas_sim.groupby('es_fin_semana')['cantidad'].mean())

es_fin_semana
False     9.048263
True     12.533654
Name: cantidad, dtype: float64


In [18]:
from dotenv import load_dotenv
import os
from sqlalchemy import create_engine

load_dotenv()

# Necesitas un .env apuntando a intelistock_db, o ajustar la conexion aqui directo
engine = create_engine(
    f"postgresql://{os.getenv('INTELISTOCK_DB_USER')}:{os.getenv('INTELISTOCK_DB_PASSWORD')}@{os.getenv('INTELISTOCK_DB_HOST')}:{os.getenv('INTELISTOCK_DB_PORT')}/{os.getenv('INTELISTOCK_DB_NAME')}"
)

print("Conexion lista")

Conexion lista


In [19]:
from sqlalchemy import text

with engine.connect() as conn:
    for _, row in df_ventas_sim.iterrows():
        # Insertar la venta (transacción) y obtener su id
        resultado = conn.execute(
            text("INSERT INTO ventas (negocio_id, fecha) VALUES (:negocio_id, :fecha) RETURNING id"),
            {"negocio_id": 1, "fecha": row['fecha']}
        )
        venta_id = resultado.fetchone()[0]
        
        # Insertar el detalle de esa venta
        conn.execute(
            text("""INSERT INTO venta_detalle (venta_id, producto_id, cantidad, precio_unitario) 
                     VALUES (:venta_id, :producto_id, :cantidad, :precio_unitario)"""),
            {
                "venta_id": venta_id,
                "producto_id": row['producto_id'],
                "cantidad": row['cantidad'],
                "precio_unitario": row['precio_unitario']
            }
        )
    
    conn.commit()

print("Datos cargados")

Datos cargados


In [17]:
load_dotenv(override=True)
print(os.getenv('INTELISTOCK_DB_NAME'))

intelistock_db
